In [1]:
import ccxt, pandas as pd, time
from datetime import datetime, timezone

# ---------- Réglages ----------
EXCHANGE_ID = 'binance'
SYMBOL      = 'BTC/USDT'
TIMEFRAME   = '1d'
SINCE       = '2019-01-01'   # date de départ (modifiable)
CSV_PATH    = '../data/btc_spot_1d.csv'

# ---------- Exchange ----------
ex = getattr(ccxt, EXCHANGE_ID)({'enableRateLimit': True})
ex.load_markets()

# since -> ms epoch
since_ms = int(pd.Timestamp(SINCE, tz='UTC').timestamp()*1000)

# ---------- Téléchargement OHLCV en boucle ----------
rows = []
while True:
    batch = ex.fetch_ohlcv(SYMBOL, timeframe=TIMEFRAME, since=since_ms, limit=1000)
    if not batch:
        break
    rows += batch
    since_ms = batch[-1][0] + 1
    time.sleep(ex.rateLimit/1000)

# ---------- DataFrame ----------
cols = ['timestamp','open','high','low','close','volume']
df = pd.DataFrame(rows, columns=cols)
df['date'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)

# ---------- Variables quantitatives ----------
df['ret']      = df['close'].pct_change()                         # continue
df['range']    = df['high'] - df['low']                           # continue
df['day']      = df['date'].dt.day                                # discrète
df['month']    = df['date'].dt.month                              # discrète
df['weekday']  = df['date'].dt.weekday                            # discrète (0=lundi)
df['vol_10d']  = df['ret'].rolling(10).std() * (252**0.5)         # vol annualisée approx.

# ---------- Variables qualitatives ----------
# Nominale: tendance (hausse/baisse) par rapport à MM20
mm20 = df['close'].rolling(20).mean()
df['trend'] = (df['close'] > mm20).map({True: 'Hausse', False: 'Baisse'})

# Ordinale: niveau de volatilité (seuils ajustables)
def vol_bucket(x):
    if pd.isna(x): return pd.NA
    if x < 0.4:    return 'Faible'
    if x < 0.8:    return 'Moyenne'
    return 'Élevée'
df['vol_level'] = df['vol_10d'].apply(vol_bucket)

# Nominale: saison (libellés lisibles)
mois = {1:'Jan',2:'Fév',3:'Mar',4:'Avr',5:'Mai',6:'Juin',7:'Juil',8:'Août',9:'Sep',10:'Oct',11:'Nov',12:'Déc'}
df['month_name'] = df['month'].map(mois)

# ---------- Nettoyage & export ----------
df = df.dropna(subset=['open','high','low','close','volume'])
df.to_csv(CSV_PATH, index=False)
print(f"[OK] Écrit : {CSV_PATH}  ({len(df)} lignes)")


[OK] Écrit : ../data/btc_spot_1d.csv  (2491 lignes)
